In [40]:
#!pip install pandas numpy word2number collections
from word2number import w2n
import pandas as pd
import numpy as np
import unicodedata
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

## **Estructura a seguir**
1. ***Limpieza de datos***
   - Eliminar duplicador
   - Tratar los valores nulos
   - Estandarizar el formato
3. ***aplicar el analisis***
   - Resumen estadistico
       - Media
       - Moda
       - Distribucion de ratings

In [2]:
datos = pd.read_csv('resenas_flowapp.csv', sep=',')

In [3]:
df = (
    datos
    .copy()
        .drop_duplicates()
     )

## **Limpieza de datos**
Principalmente se limpio el formato del texto para quitar los emojis, y texto caracteristico (tildes)

In [54]:
# normalizar texto

def limpiar_texto(texto):
    if isinstance(texto, str):
        # Eliminar tildes y emojis
        texto = (
            unicodedata.normalize("NFKD", texto)
            .encode("ascii", "ignore")
            .decode("utf-8")
        )
        return texto
    return texto

#limpiar el rating

def limpiar_rating(valor):
    
    # convertir los valores anormales o nules que se toman como texto a NaN tratable 
    if pd.isna(valor) or str(valor).strip() in ["", "?", "nan", "None"]:
        return np.nan

    # convertir a string y quitar los espacios
    val_str = str(valor).strip().lower()

    # Intento A: Si es un dígito directo ("213", "-1", "5")
    try:
        # si es un numero ya permitido transformarlo a flotante
        numero = float(val_str)
        
    except ValueError:
        # transformar de texto a numero directo
        try:
            numero = float(w2n.word_to_num(val_str))
        except ValueError:
            # Si no se puede convertir, pasarlo a entero
            return np.nan

       # para tratar valores que estan fuera del rango o con valores diferentes
    if numero < 1:
        return 1
    elif numero > 5:
        return 5
    else:
        return numero

def palabras_frecuentes_por_rating(df, top_n=3):
    print("--- PALABRAS MÁS FRECUENTES POR RATING ---")
    
    # Lista de palabras vacías comunes a ignorar (stopwords)
    stopwords = {'y', 'el', 'la', 'de', 'que', 'un', 'una', 'muy', 'pero', 'no', 'me', 'sin', 'ni', 'a', 'app', 'las',',','en','los', 'es'}
    
    for rating, grupo in df.groupby('rating'):
        # Unir todo el texto del grupo y extraer palabras
        todas_las_palabras = " ".join(grupo['texto']).split()
        # Filtrar stopwords
        palabras_filtradas = [p for p in todas_las_palabras if p not in stopwords]
        
        # Contar frecuencias
        conteo = Counter(palabras_filtradas)
        mas_comunes = conteo.most_common(top_n)
        
        print(f"\nRating {rating}:")
        for palabra, frec in mas_comunes:
            print(f"  - '{palabra}': {frec} veces")
# Conteo de palabras con countvectorizer para detectar las palabras mas frecuentes
def detectar_palabras(serie):
    # configuracion del vectorizer
    vectorizador = CountVectorizer(token_pattern=r'\b\w{6}\b')

    matriz = vectorizador.fit_transform(serie)

    return vectorizador.get_feature_names_out()

In [55]:
# Palabras mas comunes
array = detectar_palabras(df[df['rating']==3]['texto'])

In [64]:
array

array(['backup', 'basico', 'buenas', 'ccarga', 'cumple', 'faltaa',
       'faltta', 'genero', 'inicio', 'normal', 'oscuro', 'podria',
       'precio', 'tablet', 'vecces', 'vecess', 'widget'], dtype=object)

In [63]:
from spellchecker import SpellChecker

spell = SpellChecker(language='es')
for i in array:
    print(spell.correction(i))

None
básico
buena
carga
simple
falta
falta
género
inicio
normal
oscuro
podrida
precio
tableta
vencer
receso
None


In [59]:
!pip install pyspellchecker

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.2 MB 1.2 MB/s eta 0:00:06
   ---- ----------------------------------- 0.8/7.2 MB 1.2 MB/s eta 0:00:06
   ----- ---------------------------------- 1.0/7.2 MB 1.1 MB/s eta 0:00:06
   ------- -------------------------------- 1.3/7.2 MB 1.2 MB/s eta 0:00:06
   -------- ------------------------------- 1.6/7.2 MB 1.1 MB/s eta 0:00:05
   ---------- ----------------------------- 1.8/7.2 MB 1.1 MB/s eta 0:00:05
   ----------- ---------------------------- 2.1/7.2 MB 1.2 MB/s eta 0:00:05
   ------------- -------------------------- 2.4/7.2 MB 1.2 MB/s eta 0:00:05
   -------------- ------------------------- 2.6/7.2 MB 1.2 MB/s eta 0:00:04
   --------------- ------------------------ 2.9/7.2 MB 1.2 MB/s eta 0:00:04
   ----------------- ------------

In [18]:
# usar la funcion y convertirlo a minuscula
df["texto"] = (
    df["texto"].apply(limpiar_texto).str.lower().str.strip()
              )

# Reemplazar la descripcion (texto) con el valor sin comentario
df['texto'] = df['texto'].fillna('sin comentarios')

# Uso de la funcion para arreglar los valores atipicos o fuera del rango
df['rating'] = df['rating'].apply(limpiar_rating)

# Eliminando los valores nulos en rating para no sesgar la informacion
df = df.dropna(subset=['rating'])

In [15]:
df['fecha'] = pd.to_datetime(df['fecha'])

In [50]:
df[df['texto'].str.contains('buuena')]

,usuario,fecha,texto,rating
265,valeria552,2026-06-04,"muy buuena app, aunque el login podria mejorar...",4.0
